<a href="https://colab.research.google.com/github/MuskaanPrabhakar/learning_ai/blob/main/quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Image
uploading image, converting binary to string

In [ ]:
!pip install openai
from openai import OpenAI

In [ ]:
from google.colab import userdata

In [ ]:
client = OpenAI(
    base_url ="https://generativelanguage.googleapis.com/v1beta/",
    api_key=userdata.get("GEMINI_API_KEY")
)

In [ ]:
client1 = OpenAI(
    base_url ="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY")
)

In [ ]:
from google.colab import files

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
uploaded =files.upload()
for filename in uploaded.keys():
  img=cv2.imread(filename)
  print(img)
  cv2_imshow(img)

In [ ]:
import base64
for filename in uploaded.keys():
  with open(filename,'rb') as image_file: #rb is for reading binary
    image_bytes=image_file.read()

base64_string=base64.b64encode(image_bytes).decode("utf-8")
print(base64_string)

In [ ]:
response=client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {
        "role":"user", "content":[
            {
                "type":'text',
                "text":"describe the image"
            },
            {
                "type":'image_url',
                "image_url":{
                    "url": "f"data:image/jpeg;base64,{base64_string}"
                }
            }
        ]
    }
  ]
)
print(response.choices[0].message.content)

In [ ]:
response=client1.chat.completions.create(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    messages=[
        {
        "role":"user", "content":[
            {
                "type":'text',
                "text":"describe the image"
            },
            {
                "type":'image_url',
                "image_url":{
                    "url": "f"data:image/jpeg;base64,{base64_string}"
                }
            }
        ]
    }
  ]
)
print(response.choices[0].message.content)

#Quantization

In [ ]:
!pip install --upgrade -q  transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model_8bit = AutoModelForCausalLM.from_pretrained(
    "bigscience/bloom-1b7",
    device_map="auto",
    quantization_config= quantization_config
)

In [ ]:
from transformers import(
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    pipeline
)
model_name="bigscience/bloom-1b7"
quantization_config=BitsAndBytesConfig(load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipe =pipeline(
    "text-generation",
    model=model_8bit,
    tokenizer=tokenizer
)
result=pipe(
    "what is machine learning?",
    max_new_tokens=100
)
print(result[0]["generated_text"])

In [ ]:
import requests
from google.colab import userdata
cloudflare= userdata.get("cloudflare")
capi= userdata.get('capi')

def chat(message):
  url=f"https://api.cloudflare.com/client/v4/accounts/{cloudflare}/ai/run/@cf/meta/llama-3.1-8b-instruct"
  headers ={
      "Authorization":f"Bearer {capi}",
      "Content-Type": "application/json",
  }
  payload ={
      "messages":[
          {
              "role":"user",
              "content":message
          }
      ]
  }

  response = requests.post(url, headers=headers,json=payload)
  print("Status code : ", response.status_code)
  if response.status_code !=200:
    print(response.text)
    return None

  data = response.json()
  return data["result"]["response"]

print(chat("YOYO"))